In [13]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Annotated,Literal
from pydantic import BaseModel
from dotenv import load_dotenv
import os

In [14]:
load_dotenv(dotenv_path=".env", override=True)

True

In [15]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [16]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [17]:
class sentimentstate(TypedDict):
    review:str
    sentiment:Literal['positive', 'negative']
    diagnosis:dict
    response:str

In [18]:
def find_sentiment(state:sentimentstate) -> sentimentstate:
    prompt=f'For the following review find out the sentiment \n {state["review"]}'
    sentiment=model.invoke(prompt)
    
    return {'sentiment':sentiment}

In [19]:
def check_sentiment(state:sentimentstate) -> sentimentstate:
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'negative_response'

In [20]:
def run_diagnosis(state:sentimentstate) :
    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    diagnosis=model.invoke(prompt)
    return {'diagnosis':diagnosis}

In [21]:
def positive_response(state:sentimentstate) -> sentimentstate:
    prompt = f"""Write a positive response to this review:\n\n{state['review']}"""
    response=model.invoke(prompt)
    return {'response':response}

In [22]:
def negative_response(state:sentimentstate) -> sentimentstate:
    diagnosis=state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response=model.invoke(prompt)
    return {'response':response}

In [23]:
graph=StateGraph(sentimentstate)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)

In [ ]:
graph.add_edge(START,'find_sentiment')
graph.add_edge('find_sentiment',check_sentiment)

graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('positive_response',END)
graph.add_edge('negative_response',END)

TypeError: 'function' object is not iterable